# Predicting the dielectric constant of crystals

### A step-by-step machine learning tutorial

We want to predict $\varepsilon_\infty$, the **optical (electronic) dielectric constant** of a
crystal, from cheap descriptors such as its band gap, density and composition.

Calculating $\varepsilon_\infty$ properly needs density functional perturbation theory (DFPT),
which costs hours of CPU time per material. A trained machine learning model predicts it in
microseconds. That is the trade we are making.

**How this notebook is organised**

| Section | What we do |
|---|---|
| 1 | Load the data and look at it |
| 2 | Build a baseline so we know what "bad" looks like |
| 3 | Linear regression, the simplest real model |
| 4 | k-nearest neighbours, our first non-linear model |
| 5 | Support vector regression, with its settings left at default |
| 6 | Tune the SVR settings with a grid search |
| 7 | Compare everything |
| 8 | Look at the worst predictions and learn something chemical |

---
## 0. Setup

Nothing interesting here, just imports. We will introduce each scikit-learn tool
at the point where we first use it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score

RANDOM_STATE = 0          # so that everyone in the room gets the same numbers
plt.rcParams["figure.dpi"] = 110

---
## 1. The data

Two files:

* `raw_data.pickle` holds the feature matrices and the target values, already split
  into a training set and a test set.
* `data_train.h5` holds chemical metadata (formula, energy above hull). We use it only
  at the very end, for interpretation. **It is never used for training.**

The three lines about `pandas.core.indexes.numeric` are a compatibility patch. The pickle
was written with an older pandas that had a class which no longer exists, so we register a
stand-in before unpickling. This is a good illustration of why a pickle is *not* an archival
data format.

In [ ]:
!wget -q -O raw_data.pickle 'https://raw.githubusercontent.com/WMD-group/Dielectric_ML/master/dataset/raw_data.pickle'
!wget -q -O data_train.h5   'https://raw.githubusercontent.com/WMD-group/Dielectric_ML/master/dataset/data_train.h5'
print("downloaded")

In [ ]:
import sys, types, pickle
from pandas.core.indexes.base import Index

# --- compatibility patch for the old pickle ---------------------------------
shim = types.ModuleType("pandas.core.indexes.numeric")
shim.Int64Index = shim.Float64Index = shim.UInt64Index = Index
sys.modules["pandas.core.indexes.numeric"] = shim
# ----------------------------------------------------------------------------

with open("raw_data.pickle", "rb") as f:
    X_train, X_test, y_train, y_test = pickle.load(f)

store = pd.HDFStore("data_train.h5")
meta = store["df_all"]          # formulas, hull energies: for interpretation only
store.close()

print(f"training set : {X_train.shape[0]:5d} materials, {X_train.shape[1]} features")
print(f"test set     : {X_test.shape[0]:5d} materials")

### What do the features look like?

93 numbers describe each crystal. Only a handful are continuous physical descriptors
(band gap, density, formation energy, oxidation states, Madelung energies). The other 85 are
one-hot flags saying which ions are present.

Keep that imbalance in mind. It becomes important in section 5.

In [ ]:
print("first few feature names:")
print(list(X_train.columns[:8]))

X_train.iloc[:, :6].describe().round(2)

### And the target?

Always plot your target before you model it.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(y_train, bins=40, color="#3B4C9C", edgecolor="white")
ax.set_xlabel(r"$\varepsilon_\infty$")
ax.set_ylabel("number of materials")
ax.set_title("Distribution of the target in the training set")
plt.show()

print(y_train.describe().round(2))

Notice the long tail. Most materials sit between 2 and 8, and only a few reach beyond 15.

**This one plot predicts a lot of what follows.** A model trained mostly on low-$\varepsilon$
materials will be poor at high-$\varepsilon$ ones, simply because it has barely seen any.

> **Your turn.** What fraction of the training set has $\varepsilon_\infty > 12$?
> Try `(y_train > 12).mean()`.

---
## 2. A baseline: always predict the mean

Before any real model, build the dumbest one possible. It tells you what "no skill" looks
like, so you can tell whether your clever model is actually earning its keep.

We score with two numbers:

* **MAE**, the mean absolute error. Same units as $\varepsilon_\infty$, so a chemist can judge it.
* **$r^2$**, the fraction of variance explained. 1.0 is perfect, 0.0 means no better than the mean.

In [ ]:
def evaluate(model, name):
    "Fit on the training set, report MAE and r2 on both sets."
    model.fit(X_train, y_train)
    pred_tr, pred_te = model.predict(X_train), model.predict(X_test)
    row = {
        "model":     name,
        "MAE train": mean_absolute_error(y_train, pred_tr),
        "MAE test":  mean_absolute_error(y_test,  pred_te),
        "r2 train":  r2_score(y_train, pred_tr),
        "r2 test":   r2_score(y_test,  pred_te),
    }
    print(f"{name:28s}  MAE test = {row['MAE test']:.3f}   r2 test = {row['r2 test']:.3f}")
    return row, pred_te


results = []

row, _ = evaluate(DummyRegressor(strategy="mean"), "0. always predict the mean")
results.append(row)

An $r^2$ of about 0 is exactly what "no skill" means. Every model from here on has to beat this.

---
## 3. Linear regression

The simplest model that actually uses the features:

$$\varepsilon_\infty = w_1 x_1 + w_2 x_2 + \dots + w_{93} x_{93} + b$$

It finds the weights $w$ that minimise the **sum of squared errors**. One line of code.

In [ ]:
row, pred_linear = evaluate(LinearRegression(), "1. linear regression")
results.append(row)

A big jump over the baseline. But we know from physics that this cannot be the whole story:
the Penn model says $\varepsilon \approx 1 + (\hbar\omega_p / E_g)^2$, so the dependence on the
band gap is an inverse square, not a straight line. A linear model cannot express that.

**Which features does it lean on?**

In [ ]:
lin = LinearRegression().fit(X_train, y_train)
coef = pd.Series(lin.coef_, index=X_train.columns)
coef.reindex(coef.abs().sort_values(ascending=False).index).head(8).round(3)

> **Your turn.** Do the largest coefficients correspond to the quantities that appear in the
> Clausius-Mossotti and Penn models (density and band gap)? Careful: the features have very
> different units, so a large coefficient does not automatically mean a large influence.
> That is one reason we will scale the features in section 5.

---
## 4. k-nearest neighbours

Our first non-linear model, and conceptually the simplest one there is:

> To predict a new crystal, find the $k$ most similar crystals in the training set
> and average their dielectric constants.

"Similar" means close in the 93-dimensional feature space. That immediately raises a problem:
if one feature is measured in thousands and another in units, the big one dominates the
distance. So we put a `StandardScaler` in front, which rescales every feature to zero mean
and unit variance.

`make_pipeline` glues the two steps together so the scaler is always fitted on training data
only, never on the test set.

In [ ]:
knn = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5))
row, pred_knn = evaluate(knn, "2. k-nearest neighbours (k=5)")
results.append(row)

Look at the training MAE for this model, and compare it with the test MAE.

With `k=5` the model is fairly flexible: each prediction depends on only five materials.
Small `k` means memorising, large `k` means over-smoothing. Same trade-off you will meet again
in the next section, wearing different clothes.

> **Your turn.** Loop over `k = 1, 3, 5, 10, 25, 50` and plot training and test MAE against `k`.
> Where does the test error bottom out?

---
## 5. Support vector regression

Now the model this tutorial is really about.

**The idea.** Ordinary least squares punishes every error, and punishes it quadratically.
SVR instead declares a tolerance $\varepsilon$: any prediction within $\varepsilon$ of the truth
is treated as correct and costs nothing. Only the points outside that "tube" pay, and they pay
linearly. It fits the flattest function that keeps almost everything inside the tube.

Three settings control it:

| knob | meaning | too small | too large |
|---|---|---|---|
| `C` | price of leaving the tube | underfits, stays flat | overfits, chases outliers |
| `epsilon` | half-width of the tube | fits the noise | throws away real trends |
| `gamma` | reach of each training point | too coarse to resolve the pattern | isolated spikes |

**The scaler is not optional here.** The default RBF kernel is
$k(x,x') = \exp(-\gamma\|x-x'\|^2)$, a single Euclidean distance over all 93 features at once.
Unscaled features would let one column decide which crystals count as similar.

In [ ]:
svr_default = make_pipeline(StandardScaler(), SVR())      # C=1, epsilon=0.1, gamma='scale'
row, pred_svr0 = evaluate(svr_default, "3. SVR, default settings")
results.append(row)

### Seeing the knobs move

Before tuning anything, get a feel for what one knob does. We vary `C` and watch the
cross-validated score. Cross-validation splits the *training* data into 5 folds, trains on 4
and scores on the held-out one, five times over, then averages.

The test set is not touched. We spend it once, at the very end.

In [ ]:
Cs = np.logspace(-2, 3, 8)
scores = [cross_val_score(make_pipeline(StandardScaler(), SVR(C=c)),
                          X_train, y_train, cv=5, scoring="r2").mean()
          for c in Cs]

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogx(Cs, scores, "o-", color="#3B4C9C")
ax.set_xlabel("C")
ax.set_ylabel("cross-validated $r^2$")
ax.set_title("Only C is changing; epsilon and gamma stay at their defaults")
ax.grid(alpha=0.3)
plt.show()

Small `C` sits low: the model will not pay to fit the data, so it stays nearly flat.
The curve rises, plateaus, and eventually stops improving. That plateau is the useful region.

> **Your turn.** Repeat this plot for `epsilon` using `np.logspace(-3, 1, 8)`.
> What happens once epsilon is comparable to the spread of the target?

---
## 6. Tuning all three knobs together

The knobs interact: raising `gamma` makes the model more flexible, which can be partly undone
by lowering `C`. So we search them jointly rather than one at a time.

`GridSearchCV` tries every combination and keeps the best by cross-validated score.
Note that we tune `gamma` as well, which the original notebook did not do.

The `svr__` prefix tells the pipeline which step the setting belongs to.

In [ ]:
param_grid = {
    "svr__C":       np.logspace(-1, 3, 5),
    "svr__epsilon": np.logspace(-2, 0, 4),
    "svr__gamma":   np.logspace(-4, -1, 4),
}

search = GridSearchCV(
    make_pipeline(StandardScaler(), SVR()),
    param_grid, cv=5, scoring="r2", n_jobs=-1,
)
search.fit(X_train, y_train)

n = len(search.cv_results_["params"])
print(f"tried {n} combinations x 5 folds = {n * 5} fits")
print("best settings:")
for k, v in search.best_params_.items():
    print(f"   {k:14s} = {v:.4g}")
print(f"best cross-validated r2 = {search.best_score_:.3f}")

In [ ]:
row, pred_svr = evaluate(search.best_estimator_, "4. SVR, tuned")
results.append(row)

**Two things to check whenever a grid search finishes.**

1. Did the best value land on the *edge* of a grid? If so the grid was too narrow: widen it and
   rerun. The cell below checks this for you.
2. Is the chosen `epsilon` physically sensible? It should sit near the uncertainty of your
   labels. These DFPT values are converged to perhaps 0.1, so an epsilon far below that is
   asking the model to reproduce numerical noise.

In [ ]:
for key, grid in param_grid.items():
    best = search.best_params_[key]
    edge = "  <-- ON A GRID EDGE, widen the range" if best in (grid.min(), grid.max()) else ""
    print(f"{key:14s} best = {best:<10.4g} range = [{grid.min():.4g}, {grid.max():.4g}]{edge}")

---
## 7. Comparing the models

In [ ]:
table = pd.DataFrame(results).set_index("model").round(3)
table["gap (test/train MAE)"] = (table["MAE test"] / table["MAE train"]).round(1)
table

Read this table in two directions.

**Down the "MAE test" column**: does added complexity actually buy accuracy?

**Across each row**: the gap between training and test error. A model that reproduces its
training data far better than anything new has *overfitted*. A large gap is a warning, not a
verdict, but it means the training numbers tell you nothing about future performance.

### The parity plot

Never quote a metric without looking at this. A single number cannot tell you *where* the
errors live.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
lims = [0, 22]
ax.plot(lims, lims, color="#5C6479", lw=1, zorder=1)          # the y = x line
ax.scatter(y_test, pred_svr, s=14, color="#3B4C9C", alpha=0.7, zorder=2)
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_aspect("equal")
ax.set_xlabel(r"$\varepsilon_\infty$  reference (DFPT)")
ax.set_ylabel(r"$\varepsilon_\infty$  predicted (SVR)")
ax.set_title("Test set: predicted against reference")
plt.show()

Points on the diagonal are perfect. Vertical distance from it is the error.

The scatter is tight below about 10 and widens sharply above it, and most high-$\varepsilon$
points sit *below* the line. Two reasons, and they reinforce each other:

* the training set holds few high-$\varepsilon$ materials, so the model is extrapolating;
* high-$\varepsilon$ compounds tend to be small-gap ones, whose DFPT values are the least
  converged in the first place.

> **Your turn.** Colour the points by band gap
> (`c=X_test["MP_band_gap"]` if that column exists, otherwise inspect `X_test.columns`).
> Do the worst predictions cluster at low gap?

---
## 8. The ten worst predictions

This is the most useful cell in the notebook, and the one most often skipped.
Reading your failures is the fastest way to understand a model.

In [ ]:
worst = pd.DataFrame({"predicted": pred_svr}, index=X_test.index).join(meta, how="inner")
worst["error"] = worst["predicted"] - worst["PG18PI17_e_electronic_ave"]

(worst.reindex(worst["error"].abs().sort_values(ascending=False).index)
      .head(10)[["MP_pretty_formula", "predicted",
                 "PG18PI17_e_electronic_ave", "error", "MP_e_above_hull"]]
      .rename(columns={"MP_pretty_formula": "formula",
                       "PG18PI17_e_electronic_ave": "reference",
                       "MP_e_above_hull": "E above hull"})
      .round(2)
      .reset_index(drop=True))

### The twist

Every one of these sits on or very near the convex hull, so they are stable, synthesisable
compounds, not computational artefacts. The model is confidently wrong about real materials.

But is it the model that is wrong?

The authors of the original study recalculated these compounds with a much denser
Brillouin-zone mesh. For **eight of the ten**, the machine learning prediction turned out to be
*closer to the carefully converged value than the training label was*:

| | SVR | dataset label | converged DFPT |
|---|---|---|---|
| Ga$_2$Te$_5$ | 10.48 | 17.19 | 13.75 |
| LiAsS$_2$ | 7.05 | 12.39 | 8.57 |

The mechanism is specific. Small-gap materials need a dense k-mesh for a linear-response
calculation. The high-throughput workflow that generated the dataset used a standardised mesh,
so exactly those materials came out under-converged. A model that has learned the general
physical trend disagrees with individual bad labels.

**The lesson: a large residual is a data-quality signal at least as often as a model-quality
signal.**

---
## 9. Where you should refuse to use this model

Every model has a domain. State it plainly:

* materials with a band gap below 0.5 eV were removed from the training data, so the model has
  never seen one;
* materials with $\varepsilon_\infty$ above roughly 12 are sparsely represented, and the parity
  plot shows the errors there;
* the features come from Materials Project and pymatgen, so a hypothetical structure not in
  those databases needs its descriptors computed the same way.

---
## Exercises

**Warm-up**

1. How many support vectors does the tuned SVR use? Try
   `len(search.best_estimator_.named_steps["svr"].support_)` and compare it with the number of
   training materials. What fraction was discarded?
2. Refit the SVR with `epsilon=1.0` and count the support vectors again.
3. Remove the `StandardScaler` from the SVR pipeline and rerun. How much does the test MAE change?

**Core**

4. Add `RandomForestRegressor` to the comparison. Does an ensemble beat the kernel method here?
5. Plot a learning curve: train on 10%, 25%, 50%, 75% and 100% of the training set and plot
   test MAE against training size. Is it still improving at the right-hand edge? What does that
   tell you about whether more data would help?
6. Use `search.cv_results_` to plot mean train score against mean test score across the grid.
   Where does overfitting begin?

**Extension**

7. Install `shap` and run it on the tuned model. Are density and band gap the most important
   features, as the Clausius-Mossotti and Penn models suggest?
8. Restrict the training set to oxides only and see whether a narrower model does better on
   oxides than the general one.

---

### One thing to take away

Machine learning here is not a black box that replaced the physics. It is a fast interpolator
whose failures are informative. It reproduces the trends the textbook models describe, it
predicts in microseconds what DFPT needs hours for, and its largest errors point at entries in
the training database that deserve a second look.